# ✂️🎬️ Visual Understanding Playground

<a href="https://colab.research.google.com/github/video-db/videodb-cookbook/blob/main/guides/scene-index/playground_scene_extraction.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

> This guide compares separate visual-understanding experiments before creating one final semantic index.

The right segmentation and frame sampling depend on the video: a podcast may need fewer visual ranges than a fast-moving sports clip.

Inspect each experiment's stored artifact directly. Only the final scene-description experiment is indexed for retrieval.

## Setup
---

### 📦  Installing packages   

In [1]:
!pip install -q videodb


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 86.5/86.5 kB 5.3 MB/s eta 0:00:00


### 🔑 API Keys

In [2]:
import videodb
import os
from getpass import getpass

api_key = getpass("Please enter your VideoDB API Key: ")

os.environ["VIDEO_DB_API_KEY"] = api_key

Please enter your VideoDB API Key: ··········


### 🌐 Connect to VideoDB

In [3]:
from videodb import connect

conn = connect()
coll = conn.get_collection()


### 🎥  Upload Video

In [4]:
video = coll.upload(url="https://www.youtube.com/watch?v=LejnTJL173Y")

## ✂️🎬 Explore Visual Understanding Artifacts
---

Run separate time-based and shot-based understanding experiments, then inspect their timestamped artifacts before choosing a final retrieval design.

### ⚙️ Understanding Parameters

- `segmentation` controls the timestamped visual ranges for one understanding run.
- Analyzer `sampling` controls which frames a visual analyzer examines within each range.

The following experiments keep these choices separate so their outputs can be compared directly.

### **⚙️ Time-Based Sampling**

This experiment creates fixed 30-second ranges and samples three uniformly distributed frames from each range. It is useful for comparing broad, evenly spaced visual coverage.

In [5]:
print("Creating 30-second visual descriptions...")

time_understanding = video.understand(
    segmentation={"type": "time", "seconds": 30},
    analyzers=[
        {
            "type": "vlm",
            "name": "time_sampling",
            "sampling": {
                "strategy": "uniform",
                "frame_count": 3,
            },
        },
    ],
)
time_understanding.wait_until_complete()

time_analyzer = time_understanding.get_analyzer("time_sampling")


Creating 30-second visual descriptions...


### **⚙️ Shot-Based Sampling**

This separate experiment finds visual shot changes with threshold `15` and samples five frames from each shot. It emphasizes transitions, lighting changes, and movement.

In [6]:
print("Creating shot-based visual descriptions...")

shot_understanding = video.understand(
    segmentation={"type": "shot", "threshold": 15},
    analyzers=[
        {
            "type": "vlm",
            "name": "shot_sampling",
            "sampling": {
                "strategy": "uniform",
                "frame_count": 5,
            },
        },
    ],
)
shot_understanding.wait_until_complete()

shot_analyzer = shot_understanding.get_analyzer("shot_sampling")


Creating shot-based visual descriptions...


## Inspecting Stored Artifacts

Wait for each understanding run and analyzer to succeed before reading its raw artifact. The preview below keeps the server-provided scene envelope intact.

In [7]:
import json


def print_artifact_preview(analyzer, label):
    output = analyzer.get_output()
    scenes = output["scenes"]
    first_scene = scenes[0]

    print(f"{label}: {len(scenes)} scenes")
    print(json.dumps(first_scene, indent=2))
    return output


print_artifact_preview(time_analyzer, "30-second sampling")


30-second sampling: 18 scenes
{
  "data": {
    "text": "Time 0.0\u201330.0s \u2014 three main shots:\n\n- Shot 1 (early in the range): medium close-up of a middle\u2011aged, balding man wearing a dark suit, light shirt and striped tie. He is seated indoors in front of horizontal window blinds; the lighting is soft and even. His expression is serious and restrained, as if speaking in a measured way.\n\n- Shot 2: wider exterior/frame under a canopy or tent showing a standing audience. Several people in the back row are visible in full light (two men in jackets, others in casual shirts); many people in the foreground are seated and out of focus. A few audience members have raised hands; some appear to be clapping. The scene suggests a public gathering or ceremony.\n\n- Shot 3: almost the same framing as shot 2 but slightly adjusted \u2014 back row of standing attendees remains in focus, foreground seats remain blurred, and a woman at the right looks like she is praying or clasping her ha

{'metadata': {},
 'name': 'time_sampling',
 'scenes': [{'data': {'text': 'Time 0.0–30.0s — three main shots:\n\n- Shot 1 (early in the range): medium close-up of a middle‑aged, balding man wearing a dark suit, light shirt and striped tie. He is seated indoors in front of horizontal window blinds; the lighting is soft and even. His expression is serious and restrained, as if speaking in a measured way.\n\n- Shot 2: wider exterior/frame under a canopy or tent showing a standing audience. Several people in the back row are visible in full light (two men in jackets, others in casual shirts); many people in the foreground are seated and out of focus. A few audience members have raised hands; some appear to be clapping. The scene suggests a public gathering or ceremony.\n\n- Shot 3: almost the same framing as shot 2 but slightly adjusted — back row of standing attendees remains in focus, foreground seats remain blurred, and a woman at the right looks like she is praying or clasping her hands

In [8]:
print_artifact_preview(shot_analyzer, "shot-based sampling")

shot-based sampling: 84 scenes
{
  "data": {
    "text": "- A middle-aged man in a dark suit, light-colored shirt and a diagonally striped tie sits centered in frame against a backdrop of horizontal window blinds and glass office partitions. The shot is a medium close-up from about the chest up.\n\n- Frame 1 (start): He is leaning back slightly, head tilted to his left, mouth open as if mid-sentence. Expression is engaged, attentive.\n\n- Frame 2: His head turns a little to his right and his eyes glance off to the side; eyebrows lift slightly and his lips form a rounded shape, continuing to speak.\n\n- Frame 3: He drops his gaze and his eyelids lower; expression becomes more reflective and subdued, as if considering what he just said.\n\n- Frame 4: Similar posture to frame 3, head slightly down, mouth pursed; the mood reads serious and thoughtful.\n\n- Frame 5 (end of range): He brings his head back to a more neutral, forward position. Expression relaxes into a calm, closed-mouth look 

{'metadata': {},
 'name': 'shot_sampling',
 'scenes': [{'data': {'text': '- A middle-aged man in a dark suit, light-colored shirt and a diagonally striped tie sits centered in frame against a backdrop of horizontal window blinds and glass office partitions. The shot is a medium close-up from about the chest up.\n\n- Frame 1 (start): He is leaning back slightly, head tilted to his left, mouth open as if mid-sentence. Expression is engaged, attentive.\n\n- Frame 2: His head turns a little to his right and his eyes glance off to the side; eyebrows lift slightly and his lips form a rounded shape, continuing to speak.\n\n- Frame 3: He drops his gaze and his eyelids lower; expression becomes more reflective and subdued, as if considering what he just said.\n\n- Frame 4: Similar posture to frame 3, head slightly down, mouth pursed; the mood reads serious and thoughtful.\n\n- Frame 5 (end of range): He brings his head back to a more neutral, forward position. Expression relaxes into a calm, cl

## Viewing, Inspecting, and Deleting Understanding Runs
---

Each understanding run stores named analyzer artifacts for the video.

Use the run APIs to list, reopen, inspect, and delete an understanding only when none of its artifacts are needed later.

**Viewing all understanding runs for a video**:

In [9]:
understandings = video.list_understandings()
for understanding in understandings:
    print(f"Understanding ID: {understanding.id}, status: {understanding.status}")

Understanding ID: und_030bf4dff0cf469f, status: done
Understanding ID: und_5625952be00b476b, status: done


**Get an understanding run by ID**:

In [10]:
first_understanding = video.get_understanding(time_understanding.id)
print(first_understanding)

Understanding(id=und_030bf4dff0cf469f, video_id=m-z-019f9348-0e66-71c1-892e-19642308a24e, status=done, analyzers=1)


**Inspecting analyzer handles in an understanding run**:

In [11]:
print(f"Understanding ID: {shot_understanding.id}")
for analyzer in shot_understanding.list_analyzers():
    print(f"{analyzer.name}: {analyzer.type} ({analyzer.status})")

Understanding ID: und_5625952be00b476b
shot_sampling: vlm (done)


**Delete an understanding run that is no longer needed**:

In [12]:
# This shot-sampling run has already been inspected and is not used later.
shot_understanding.delete()

## ✍️ Playground: Play with Prompt
---

Before finalizing a retrieval design, experiment with prompts on separate artifacts and inspect their raw outputs.

The single-frame and multi-frame prompt experiments below remain independent so each can be evaluated on its own terms.

### Single-Frame Object Prompt

In [13]:
frame_prompt = """
You will be provided with an image. Your task is to identify and describe the objects in the image.
1.	Identify Objects: List distinct objects in the image.
2.	Describe Objects: Provide a brief description of each object, including shape, color, and any notable features.

Ouput should be a list of objects
Expected Output:
[{"name": "book", "context": "a person wearing a white shirt is holding a book"}]
"""

single_frame_understanding = video.understand(
    segmentation={"type": "time", "seconds": 30},
    analyzers=[
        {
            "type": "vlm",
            "name": "frame_prompt_experiment",
            "sampling": {
                "strategy": "uniform",
                "frame_count": 1,
            },
            "config": {"prompt": frame_prompt},
        },
    ],
)
single_frame_understanding.wait_until_complete()

single_frame_analyzer = single_frame_understanding.get_analyzer(
    "frame_prompt_experiment"
)

print_artifact_preview(single_frame_analyzer, "single-frame object prompt")


single-frame object prompt: 18 scenes
{
  "data": {
    "value": [
      {
        "context": "a crowd of seated and standing people, various ages, mostly facing forward; some have raised hands; clothing includes shirts, jackets, and dresses in muted colors",
        "name": "people"
      },
      {
        "context": "a teal-colored fabric canopy overhead with a straight horizontal edge providing shade",
        "name": "canopy/tent roof"
      },
      {
        "context": "thin vertical poles holding up the canopy, visible at center and sides",
        "name": "support poles"
      },
      {
        "context": "a small dark-colored book or notebook held by a standing person near the center of the image",
        "name": "book"
      },
      {
        "context": "green grassy field visible behind the standing group, stretching toward the background",
        "name": "grass/field"
      },
      {
        "context": "a line of bare, thin-branched trees in the distant background aga

{'metadata': {},
 'name': 'frame_prompt_experiment',
 'scenes': [{'data': {'value': [{'context': 'a crowd of seated and standing people, various ages, mostly facing forward; some have raised hands; clothing includes shirts, jackets, and dresses in muted colors',
      'name': 'people'},
     {'context': 'a teal-colored fabric canopy overhead with a straight horizontal edge providing shade',
      'name': 'canopy/tent roof'},
     {'context': 'thin vertical poles holding up the canopy, visible at center and sides',
      'name': 'support poles'},
     {'context': 'a small dark-colored book or notebook held by a standing person near the center of the image',
      'name': 'book'},
     {'context': 'green grassy field visible behind the standing group, stretching toward the background',
      'name': 'grass/field'},
     {'context': 'a line of bare, thin-branched trees in the distant background against the sky',
      'name': 'leafless trees'},
     {'context': 'diffuse daylight illuminat

### Multi-Frame Scene Prompt

In [14]:
scene_prompt = """
You will be provided with a series of images. Your task is to view all images together and describe the overall story or scene in the best possible way.

Expected Output:
- A detailed story or scene description.
- A list of objects and actions in each image.

Example Output:
{
  "scene_story": "A person is cooking in the kitchen and then someone rings the doorbell.",
  "images": [
    {"description": "Someone is cooking in the kitchen."},
    {"description": "Someone rings the doorbell."}
  ]
}
"""

scene_prompt_understanding = video.understand(
    segmentation={"type": "time", "seconds": 30},
    analyzers=[
        {
            "type": "vlm",
            "name": "scene_prompt_experiment",
            "sampling": {
                "strategy": "uniform",
                "frame_count": 3,
            },
            "config": {"prompt": scene_prompt},
        },
    ],
)
scene_prompt_understanding.wait_until_complete()

scene_prompt_analyzer = scene_prompt_understanding.get_analyzer(
    "scene_prompt_experiment"
)

print_artifact_preview(scene_prompt_analyzer, "multi-frame scene prompt")


multi-frame scene prompt: 18 scenes
{
  "data": {
    "images": [
      {
        "actions": [
          "sitting",
          "speaking",
          "listening/answering"
        ],
        "description": "A middle-aged man in a dark suit and striped tie sits in front of a window with horizontal blinds. He has a serious expression and is mid-sentence, suggesting he is speaking or being interviewed in an office-like setting.",
        "objects": [
          "suit",
          "tie",
          "shirt",
          "horizontal blinds",
          "chair",
          "window"
        ]
      },
      {
        "actions": [
          "standing",
          "applauding",
          "raising hands",
          "watching/listening"
        ],
        "description": "An outdoor scene under a large tent: a crowd of people, some seated in the foreground (out of focus) and a line of people standing at the back. Several in the standing group have raised hands or are applauding; the mood is attentive and emo

{'metadata': {},
 'name': 'scene_prompt_experiment',
 'scenes': [{'data': {'images': [{'actions': ['sitting',
       'speaking',
       'listening/answering'],
      'description': 'A middle-aged man in a dark suit and striped tie sits in front of a window with horizontal blinds. He has a serious expression and is mid-sentence, suggesting he is speaking or being interviewed in an office-like setting.',
      'objects': ['suit',
       'tie',
       'shirt',
       'horizontal blinds',
       'chair',
       'window']},
     {'actions': ['standing',
       'applauding',
       'raising hands',
       'watching/listening'],
      'description': 'An outdoor scene under a large tent: a crowd of people, some seated in the foreground (out of focus) and a line of people standing at the back. Several in the standing group have raised hands or are applauding; the mood is attentive and emotionally engaged.',
      'objects': ['tent/canopy',
       'chairs',
       'trees/field in background',
  

### 🗂️ Create a Final Scene Description Index
---

Create a separate, schema-defined scene-description artifact for retrieval. This final lesson does not reuse either exploratory prompt artifact.

> Index only the final analyzer handle. The server keeps the artifact as the source of truth, so its output does not need to be reconstructed in the client.

In [15]:
print("Creating the final scene-description artifact...")

final_scene_understanding = video.understand(
    segmentation={"type": "shot", "threshold": 15},
    analyzers=[
        {
            "type": "vlm",
            "name": "final_scene_description",
            "sampling": {
                "strategy": "uniform",
                "frame_count": 5,
            },
            "config": {
                "prompt": "Describe the scene in detail.",
                "schema": {"scene_description": "string"},
            },
        },
    ],
)
final_scene_understanding.wait_until_complete()

final_scene_analyzer = final_scene_understanding.get_analyzer(
    "final_scene_description"
)


Creating the final scene-description artifact...


### Build the Final Scene Index

Create the semantic index from the final scene-description artifact.


In [16]:
scene_index = video.index(
    name="My Custom Annotations#1",
    source=final_scene_analyzer,
    use_for=["semantic"],
    fields={"semantic": ["scene_description"]},
)

scene_index.wait_until_complete()

index_id = scene_index.index_id
print(scene_index)


Index(index_id=89c9b568327e4815, video_id=m-z-019f9348-0e66-71c1-892e-19642308a24e, name=My Custom Annotations#1, status=ready, use_for=['semantic'], record_count=84)


### 🔍 Search
---

The final semantic index is ready before this direct search runs.

In [17]:
# Search only the final scene-description index.
res = video.semantic_search(query="guns", index_ids=[index_id], top_k=10)
res.play()

---
If you have any questions or feedback. Feel free to reach out to us 🙌🏼

* [Discord](https://colab.research.google.com/corgiredirector?site=https%3A%2F%2Fdiscord.gg%2Fpy9P639jGz)
* [GitHub](https://github.com/video-db)
* [VideoDB](https://colab.research.google.com/corgiredirector?site=https%3A%2F%2Fvideodb.io)
* [Email](ashu@videodb.io)